# 01 Data ingestion — Victorian recorded offences

## Objective
Load official Crime Statistics Agency (CSA) Excel releases into a reproducible SQLite analysis database.

## Data source
- Dataset: CSA recorded offences, year ending March 2026
- Catalogue: https://discover.data.vic.gov.au/dataset/data-tables-recorded-offences
- Publisher: Crime Statistics Agency, Victoria
- Licence: Creative Commons Attribution 4.0
- Reporting period: year ending March 2017 to year ending March 2026

## Business questions
- What official tables are available?
- Can the Excel releases be converted into analysis-ready tables without fabricating columns?

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
from src.config import SOURCE_FILES, SQLITE_PATH, source_path
from src.ingestion.load_csa import ingest_all

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## Data loading

In [ ]:
for key, meta in SOURCE_FILES.items():
    path = source_path(key)
    print(f"{key}: {path.name} ({path.stat().st_size/1e6:.1f} MB)")
    print(f"  dataset: {meta['dataset_name']}")
    print(f"  url: {meta['catalogue_url']}")

In [ ]:
# Re-run ingestion only if the database is missing.
if SQLITE_PATH.exists():
    print(f'Using existing database: {SQLITE_PATH}')
else:
    tables = ingest_all()
    for name, df in tables.items():
        print(name, len(df))

## Data inspection

In [ ]:
import sqlite3
conn = sqlite3.connect(SQLITE_PATH)
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
display(tables)
for name in tables['name']:
    info = pd.read_sql_query(f'PRAGMA table_info({name})', conn)
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {name}', conn)['n'][0]
    print(f'\n{name}: {n:,} rows')
    display(info[['name','type']])
conn.close()

## Findings
The LGA workbook contains LGA totals, offence-type-by-LGA, location type and investigation status. The visualisation workbook contains statewide offence type, family-incident flag and investigation status. Suburb-level Table 03 is excluded from the first analysis layer because it is a 370,000-row location extract, not required for state/LGA statistical products.

## Limitations
Recorded offences are not unique criminal incidents, alleged offender incidents, or proven offences. Sensitive counts of 3 or fewer for homicide and sexual-offence subdivisions are confidentialised by CSA.